# ECGData LFCC + temporal Swin: manuscript workflow

An executable research record in manuscript order: cohort, patient split, LFCC construction, model, training, controls, evaluation, uncertainty, and follow-up studies. The default run verifies and reuses completed artifacts; set `ECG_RUN_FRESH=1` to train the original protocol into a **new** output directory. This notebook requires the project checkout and local gitignored data and artifacts. It does not require or process ACS/OMI waveforms.

## 1. Environment and reproducibility

In [1]:
import os, sys, json, time
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
here = Path.cwd().resolve()
REPO = next(p for p in (here, *here.parents) if (p / 'attention method' / 'ecgdata_protocol.json').exists())
ATT = REPO / 'attention method'
sys.path.insert(0, str(ATT))
sys.path.insert(0, str(ATT / '.venv/torch-cu128'))
import numpy as np, pandas as pd, torch
from IPython.display import display
from attention_ecg.ecgdata import verify_cache, read_json, file_hash, matched_windows
from attention_ecg.ecgdata_training import partition, verify_trial, run_training
from attention_ecg.ecgdata_report import collect_predictions, patient_confusions, scores, bootstrap_comparison
print('Project:', REPO)
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Project: /mnt/d/Projects/test-ecg-training
Python: 3.12.3 | NumPy: 2.5.2 | PyTorch: 2.7.1+cu128
CUDA available: True


## 2. Cohort, labels and patient-separated folds
Each patient and all their recordings/leads stay in one fold. The cohort has 162 lead rows from 81 recordings and 80 patients. ARR, CHF and NSR originate from different sources; diagnosis and source are confounded.

In [2]:
cache_manifest, data = verify_cache(ATT / 'features/ecgdata_lfcc_v1')
protocol = cache_manifest['protocol']
classes = protocol['classes']
print('Feature array:', data['X'].shape, '| patients:', len(np.unique(data['patients'])))
df = pd.DataFrame({'fold': data['fold'], 'patient': data['patients'], 'label': [classes[i] for i in data['y']]})
for f in range(5):
    tr, te = partition(data, f)
    assert not set(data['patients'][tr]) & set(data['patients'][te])
print('Patient overlap across each train/test boundary: 0')
display(df.groupby(['fold','label']).agg(windows=('patient','size'), patients=('patient','nunique')).reset_index())

Feature array: (1620, 13, 1, 16) | patients: 80
Patient overlap across each train/test boundary: 0


,fold,label,windows,patients
0,0,ARR,200,9
1,0,CHF,60,3
2,0,NSR,80,4
3,1,ARR,180,9
4,1,CHF,60,3
5,1,NSR,80,4
6,2,ARR,200,10
7,2,CHF,60,3
8,2,NSR,60,3
9,3,ARR,180,9


## 3. Signal processing and LFCC features
Each 500-sample window is standardized, divided into one-second frames, and represented by 16 linear-frequency cepstral coefficients per frame. Normalization is fitted only on the training patients.

In [3]:
from attention_ecg.cepstral import CepstralConfig, cepstral_features
windows, aligned = matched_windows(REPO / 'ECGData.mat', protocol)
cfg = CepstralConfig(**protocol['cepstral'])
feature = cepstral_features(windows[0][:,None].astype(np.float64), cfg)[0].astype(np.float32)
np.testing.assert_allclose(feature, data['X'][0], rtol=1e-5, atol=1e-5)
print('Raw standardized window:', windows[0].shape, '| LFCC:', feature.shape)
print('Sampling rate: 128 Hz; frames/hop: 128/32; FFT: 256; 32 linear bands; c0–c15.')
print('First window reproduces saved LFCC cache.')

Raw standardized window: (500,) | LFCC: (13, 1, 16)
Sampling rate: 128 Hz; frames/hop: 128/32; FFT: 256; 32 linear bands; c0–c15.
First window reproduces saved LFCC cache.


## 4. Temporal Swin and control
The Swin is a small, randomly initialized 1-D adaptation; it is not a pretrained image model. The classical control uses the means and standard deviations of LFCCs with logistic regression.

In [4]:
from attention_ecg.model import CepstralSwin
model = CepstralSwin(**protocol['model'])
print('Swin parameters:', sum(x.numel() for x in model.parameters()))
print('Architecture:', json.dumps(protocol['model'], indent=2))
print('Control:', json.dumps(protocol['control'], indent=2))

Swin parameters: 33607
Architecture: {
  "n_leads": 1,
  "n_ceps": 16,
  "num_classes": 3,
  "embed_dim": 16,
  "depths": [
    2,
    2
  ],
  "heads": [
    2,
    4
  ],
  "window_size": 4
}
Control: {
  "kind": "temporal mean and population std of cepstra -> StandardScaler -> logistic regression",
  "C": 1.0,
  "class_weight": "balanced",
  "max_iter": 2000,
  "solver": "lbfgs"
}


## 5. Full training entry point and completed-fit verification
The original experiment trains five patient folds × seeds 0, 1, 2 for 40 epochs, plus five logistic controls. The function below runs the complete original training procedure into a new directory when explicitly enabled. Default execution verifies all existing models and skips retraining.

In [5]:
run = ATT / 'results/ecgdata_lfcc_swin_v1'
for fold in range(5):
    assert verify_trial(run / f'logistic_fold{fold}')
    for seed in (0,1,2):
        assert verify_trial(run / f'swin_seed{seed}_fold{fold}')
print('Verified:', 15, 'Swin fits and', 5, 'logistic controls; seeds 0, 1, 2.')
if os.environ.get('ECG_RUN_FRESH') == '1':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    result = run_training(output=ATT / 'results/ecgdata_lfcc_swin_manuscript_retrain', execute=True, device=device)
    print('Fresh training:', result)
else:
    print('Using verified completed fits; set ECG_RUN_FRESH=1 for the full new training run.')

Verified: 15 Swin fits and 5 logistic controls; seeds 0, 1, 2.
Using verified completed fits; set ECG_RUN_FRESH=1 for the full new training run.


## 6. Held-out predictions and metrics
Out-of-fold predictions cover each window once per seed. Patient vote pools windows and leads for each patient. Means average seed metrics; they are not a probability ensemble.

In [6]:
predictions = collect_predictions(run, data, protocol)
rows=[]
for arm, values in predictions.items():
    for seed,pred in enumerate(values):
        w,p = patient_confusions(data['y'], pred, data['patients'])
        wm,pm = scores(w.sum(0)),scores(p.sum(0))
        rows.append(dict(model=arm, seed=seed if arm=='swin' else 'deterministic', window_accuracy=wm['accuracy'], window_macro_f1=wm['macro_f1'], patient_vote_accuracy=pm['accuracy'], patient_vote_macro_f1=pm['macro_f1']))
metrics=pd.DataFrame(rows)
display(metrics.round(4))
print('Swin means:', metrics[metrics.model=='swin'].select_dtypes('number').mean().round(4).to_dict())

,model,seed,window_accuracy,window_macro_f1,patient_vote_accuracy,patient_vote_macro_f1
0,swin,0,0.8198,0.7836,0.9250,0.9066
1,swin,1,0.8475,0.8201,0.9625,0.9586
2,swin,2,0.8333,0.7981,0.9375,0.9156
3,logistic,deterministic,0.7648,0.7342,0.8750,0.8517


Swin means: {'window_accuracy': 0.8335, 'window_macro_f1': 0.8006, 'patient_vote_accuracy': 0.9417, 'patient_vote_macro_f1': 0.9269}


## 7. Uncertainty
The paired patient bootstrap resamples patients, retaining all their windows. Intervals condition on the fitted predictions and do not include retraining or model-selection uncertainty.

In [7]:
e = protocol['evaluation']
boot = bootstrap_comparison(data['y'], data['patients'], predictions, e['bootstrap_resamples'], e['bootstrap_seed'])
print('Draws:', e['bootstrap_resamples'], '| seed:', e['bootstrap_seed'], '| discarded:', boot['rejected_single_class_draws'])
display(pd.DataFrame(boot['paired_difference_intervals_95']).T)

Draws: 2000 | seed: 20260925 | discarded: 0


,window_macro_f1,window_accuracy,patient_vote_macro_f1,patient_vote_accuracy
swin_minus_logistic,"[0.02236880952387973, 0.11427987105310342]","[0.02748636831275717, 0.11272090955284553]","[0.005497154145046393, 0.15890228744261067]","[0.004166666666666763, 0.13333333333333341]"


## 8. Completed improvements
The regularization study selected dropout/epoch within training patients. The all-window study used all 21,222 windows and compared on the same original held-out windows; its higher data volume also increased the number of optimizer updates.

In [8]:
reg = pd.read_csv(ATT / 'reports/ecgdata_lfcc_swin_regularization_v2/metrics.csv')
allw = pd.read_csv(REPO / 'experiments/ecgdata/07_lfcc_swin_all_windows/reports/metrics.csv')
print('Regularization export:', reg.shape, '| all-window export:', allw.shape)
display(reg.head(8))
display(allw.head(8))
print('Primary results: v1 83.35% window accuracy / macro-F1 0.8006; selected regularization 84.12% / 0.8099; all-window training, on v1 windows, 86.83% / 0.8390.')
print('Regularization macro-F1 difference: +0.0093 [−0.0023, +0.0212]. All-window difference: +0.0384 [+0.0153, +0.0602].')

Regularization export: (13, 11) | all-window export: (24, 10)


,arm,seed,window_accuracy,window_macro_f1,window_balanced_accuracy,patient_vote_accuracy,patient_vote_macro_f1,patient_vote_balanced_accuracy,window_recall_ARR,window_recall_CHF,window_recall_NSR
0,v1,0.0,0.819753,0.783560,0.777454,0.9250,0.906627,0.889204,0.881250,0.673333,0.777778
1,v1,1.0,0.847531,0.820126,0.816944,0.9625,0.958589,0.940741,0.895833,0.780000,0.775000
2,v1,2.0,0.833333,0.798093,0.796204,0.9375,0.915617,0.896296,0.887500,0.706667,0.794444
3,logistic,NaN,0.764815,0.734194,0.757662,0.8750,0.851716,0.860835,0.771875,0.706667,0.794444
4,R0,0.0,0.820370,0.785207,0.784722,0.9250,0.909440,0.904334,0.870833,0.683333,0.800000
5,R0,1.0,0.850617,0.823479,0.822894,0.9500,0.935600,0.918519,0.894792,0.793333,0.780556
6,R0,2.0,0.832716,0.794935,0.791065,0.9375,0.923369,0.900000,0.893750,0.693333,0.786111
7,R1,0.0,0.832716,0.802721,0.804306,0.9500,0.944281,0.933649,0.872917,0.723333,0.816667


,evaluation_set,arm,seed,scope,accuracy,macro_f1,balanced_accuracy,recall_ARR,recall_CHF,recall_NSR
0,v1_windows,all_swin,0,window,0.875309,0.848640,0.850856,0.911458,0.796667,0.844444
1,v1_windows,all_swin,0,patient_vote,0.975000,0.964745,0.959259,1.000000,0.933333,0.944444
2,v1_windows,all_swin,1,window,0.870988,0.840175,0.838310,0.917708,0.750000,0.847222
3,v1_windows,all_swin,1,patient_vote,0.962500,0.943490,0.937037,1.000000,0.866667,0.944444
4,v1_windows,all_swin,2,window,0.858642,0.828058,0.829468,0.898958,0.736667,0.852778
5,v1_windows,all_swin,2,patient_vote,0.937500,0.919116,0.907723,0.978723,0.800000,0.944444
6,v1_windows,all_logistic,deterministic,window,0.776543,0.746163,0.770208,0.782292,0.720000,0.808333
7,v1_windows,all_logistic,deterministic,patient_vote,0.900000,0.875290,0.890150,0.914894,0.866667,0.888889


Primary results: v1 83.35% window accuracy / macro-F1 0.8006; selected regularization 84.12% / 0.8099; all-window training, on v1 windows, 86.83% / 0.8390.
Regularization macro-F1 difference: +0.0093 [−0.0023, +0.0212]. All-window difference: +0.0384 [+0.0153, +0.0602].


## 9. Patient-count learning curve
The separately documented ECG-08 experiment varies training patients within each outer fold, keeping held-out patients fixed. The 100% point reuses the verified baseline.

In [9]:
learning = REPO / 'experiments/ecgdata/08_patient_learning_curve'
if (learning / 'results/completed.json').exists():
    print((learning / 'REPORT.md').read_text())
else:
    print('ECG-08 is not complete; consult its status and logs. No outcome is inferred.')

# ECG-08 patient-count learning curve

Three patient-subset draws paired with seeds 0/1/2; five patient folds; 40 epochs. Logistic uses the same subsets.

| Fraction | Swin window accuracy | Swin macro-F1 | Swin patient-vote accuracy | Logistic window accuracy |
|---|---:|---:|---:|---:|
| 25% | 75.95% | 0.7089 | 83.33% | 74.30% |
| 50% | 79.34% | 0.7490 | 89.17% | 74.96% |
| 75% | 82.18% | 0.7799 | 91.25% | 75.99% |
| 100% | 83.35% | 0.8006 | 94.17% | 76.48% |

Primary 100% minus 75% macro-F1: +0.0207, 95% patient-bootstrap interval [+0.0024, +0.0373].

Patient bootstrap conditional on predictions; subset and initialization vary together; fewer patients also means fewer optimizer steps.
The 80-patient cohort is repeatedly studied and confounds diagnosis with source. No external validation.

The primary interval is above zero under the protocol's interpretation rule,
consistent with improvement between these two training sizes on this cohort. It
is not evidence of a general learning cu

## 10. New representation experiment
ECG-09 compares 32 log-filterbank energies and a raw-waveform CNN with the original all-window LFCC–Swin baseline. Results are displayed only after its independent audit passes.

In [10]:
study = REPO/'experiments/ecgdata/09_representation_comparison'
report = study/'reports/REPORT.md'
audit = study/'reports/audit.json'
if report.exists() and audit.exists() and json.loads(audit.read_text()).get('status') == 'passed':
    print(report.read_text())
else:
    status = study/'results/status.json'
    print('ECG-09 results pending completed reporting and audit.')
    print('Last recorded study state:', json.loads(status.read_text()) if status.exists() else 'not available')


ECG-09 results pending completed reporting and audit.
Last recorded study state: {'status': 'running', 'last_completed': 'raw_cnn_seed0_fold0', 'pid': 90076, 'updated_utc': '2026-11-01T04:03:11Z'}


## 11. Interpretation and manuscript limitations
The original 80-patient cohort has been used for repeated research decisions and does not provide untouched external validation. Window and patient-vote results answer different questions. The regularization primary interval crosses zero; the all-window result combines more windows and more training updates. Numerical checks do not prove software correctness. No finding establishes clinical effectiveness, quantum advantage, or general VMD/WST superiority.

ECG-08 did not hash every transitive imported source at launch; see its report for the provenance limitation.

In [11]:
print('Source reports:')
for rel in ['attention method/reports/ecgdata_lfcc_swin_v1/results.md','attention method/reports/ecgdata_lfcc_swin_regularization_v2/SUMMARY.md','experiments/ecgdata/07_lfcc_swin_all_windows/reports/report.md','experiments/ecgdata/08_patient_learning_curve/REPORT.md']:
    print(('available' if (REPO/rel).exists() else 'pending'), rel)

Source reports:
available attention method/reports/ecgdata_lfcc_swin_v1/results.md
available attention method/reports/ecgdata_lfcc_swin_regularization_v2/SUMMARY.md
available experiments/ecgdata/07_lfcc_swin_all_windows/reports/report.md
available experiments/ecgdata/08_patient_learning_curve/REPORT.md
